# **Regression Model Exploration**

The following book explores various regression models to be used to predict the permanent magnet temperature in a PM motor.

## **Imports**

In [1]:
# Standard library imports
import pandas as pd
import numpy as np

# SkLearn preprocessing imports
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, GridSearchCV, train_test_split

# SkLearn model imports
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR

## **Data Manipulation and Preprocessing**

First we load the data, then drop unnecessary columns, and then reorder the columns to group by similarity.

In [2]:
df = pd.read_csv('data/measures_v2.csv')
df.head()

,u_q,coolant,stator_winding,u_d,stator_tooth,motor_speed,i_d,i_q,pm,stator_yoke,ambient,torque,profile_id
0,-0.450682,18.805172,19.086670,-0.350055,18.293219,0.002866,0.004419,0.000328,24.554214,18.316547,19.850691,0.187101,17
1,-0.325737,18.818571,19.092390,-0.305803,18.294807,0.000257,0.000606,-0.000785,24.538078,18.314955,19.850672,0.245417,17
2,-0.440864,18.828770,19.089380,-0.372503,18.294094,0.002355,0.001290,0.000386,24.544693,18.326307,19.850657,0.176615,17
3,-0.327026,18.835567,19.083031,-0.316199,18.292542,0.006105,0.000026,0.002046,24.554018,18.330833,19.850647,0.238303,17
4,-0.471150,18.857033,19.082525,-0.332272,18.291428,0.003133,-0.064317,0.037184,24.565397,18.326662,19.850639,0.208197,17


In [3]:
cols_to_drop = [
    'profile_id',
    # 'torque',
    'stator_tooth',
    'stator_winding'
]
columns_order = [
    'u_q', 
    'u_d', 
    'i_q', 
    'i_d',
    'torque',
    'motor_speed', 
    'ambient', 
    'coolant', 
    'stator_yoke', 
    'pm'
]

df = df.drop(cols_to_drop, axis=1)
df = df[columns_order]
df.head()

,u_q,u_d,i_q,i_d,torque,motor_speed,ambient,coolant,stator_yoke,pm
0,-0.450682,-0.350055,0.000328,0.004419,0.187101,0.002866,19.850691,18.805172,18.316547,24.554214
1,-0.325737,-0.305803,-0.000785,0.000606,0.245417,0.000257,19.850672,18.818571,18.314955,24.538078
2,-0.440864,-0.372503,0.000386,0.001290,0.176615,0.002355,19.850657,18.828770,18.326307,24.544693
3,-0.327026,-0.316199,0.002046,0.000026,0.238303,0.006105,19.850647,18.835567,18.330833,24.554018
4,-0.471150,-0.332272,0.037184,-0.064317,0.208197,0.003133,19.850639,18.857033,18.326662,24.565397


Now let's split our dataset into the feature set `X` and label set `y`.

In [4]:
X = df.drop('pm', axis=1)
y = df['pm']

Next we'll split the data into a training and test set.

In [5]:
TEST_SIZE = 0.3

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=42)

In [6]:
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

X_train shape: (931571, 9)
X_test shape: (399245, 9)


Now we'll make a pipeline that scales the data and reduces it to four principle components using PCA.

In [7]:
data_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    # ('pca', PCA(n_components=4))
])

From here you can make a pipeline to fit each model and check baseline scores:

In [9]:
models = {
    'Dummy Regressor' : DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    # 'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
    # 'SVR': SVR()
}

param_grids = {
    'Ridge': {'model__alpha': [0.1, 1.0, 10.0, 100.0]},
    'Lasso': {'model__alpha': [0.01, 0.1, 1.0, 10.0]},
    'ElasticNet': {
        'model__alpha': [0.01, 0.1, 1.0],
        'model__l1_ratio': [0.2, 0.5, 0.8]
    }
}

scoring_metrics = ['neg_mean_absolute_error', 'r2']

results = []

# Perform cross-validation for each model
for name, model in models.items():
    print(f'Running cross-validation for model: {name}')
    
    # Create model pipeline
    model_pipeline = Pipeline([
        ('data_processing', data_pipeline),
        ('model', model)
    ])
    
    # Tune hyperparameters if a parameter grid is provided
    if name in param_grids:
        grid_search = GridSearchCV(
            model_pipeline,
            param_grid = param_grids[name],
            scoring = scoring_metrics,
            refit = 'neg_mean_absolute_error',
            cv = 5
        )
        grid_search.fit(X_train, y_train)
        best_pipeline = grid_search.best_estimator_
        best_params = grid_search.best_params_
    else:
        best_pipeline = model_pipeline
        best_pipeline.fit(X_train, y_train)
        best_params = None
    
    # Perform cross-validation
    scores = cross_validate(
        best_pipeline,
        X_train,
        y_train,
        scoring=scoring_metrics,
        cv=5,
        return_train_score=True
    )

    # Store results
    results.append({
        'Model': name,
        'Train MAE': -np.mean(scores['train_neg_mean_absolute_error']),
        'Validation MAE': -np.mean(scores['test_neg_mean_absolute_error']),
        'Train R²': np.mean(scores['train_r2']),
        'Validation R²': np.mean(scores['test_r2'])
    })

# Convert results into a DataFrame
results_df = pd.DataFrame(results)
results_df

Running cross-validation for model: Dummy Regressor
Running cross-validation for model: Linear Regression
Running cross-validation for model: Ridge
Running cross-validation for model: Lasso
Running cross-validation for model: ElasticNet


,Model,Train MAE,Validation MAE,Train R²,Validation R²
0,Dummy Regressor,15.917501,15.917515,0.000000,-0.000005
1,Linear Regression,5.996245,5.996330,0.824450,0.824444
2,Ridge,5.996245,5.996331,0.824450,0.824444
3,Lasso,6.000525,6.000592,0.824333,0.824329
4,ElasticNet,6.014229,6.014299,0.824087,0.824082
